# 05 · Avaliação nos eventos extremos

Compara todos os modelos (persistência, Ridge, gradient boosting, MLP, GRU) no **período de teste** e em cada **evento** (set/2023, nov/2023, mai/2024):

- erro de **pico** (cm e %): quanto o modelo subestimou/superestimou a cota máxima;
- erro de **horário do pico** (h; positivo = previsão atrasada);
- **`cobertura_%`** (horas da janela que puderam ser avaliadas) e **`pico_perto_de_lacuna`**: se `True`, o sensor da estação falhou ao lado do pico, e o "pico observado" é menor que o real. Nesses casos o erro de pico **não** é confiável;
- acerto na passagem da **cota de alerta**, se `ALERT_LEVEL_CM` estiver preenchido em `config.py` (POD, FAR, CSI).

As previsões são alinhadas ao instante previsto (`t + h`), para comparar com a observação do mesmo momento.

In [ ]:
import os, sys
sys.path.append("..")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config, data, features, metrics, pipeline
plt.rcParams["figure.figsize"] = (11, 3.4)
plt.rcParams["axes.grid"] = True

MODELOS = ["persistencia", "ridge", "hgb", "mlp", "gru"]
FIG_DIR = config.OUTPUTS / "figs" / pipeline.current_tag(); FIG_DIR.mkdir(parents=True, exist_ok=True)
print("cenário atual:", pipeline.current_tag(), "(mude com USE_IMERG em config.py)")

def carregar(alvo):
    out = {}
    for m in MODELOS:
        try:
            out[m] = pipeline.load_predictions(m, alvo)
        except FileNotFoundError:
            print(f"[aviso] sem previsões de {m} para {alvo} (rode os notebooks 03/04)")
    return out

## 1. Desempenho geral no teste

In [ ]:
linhas = []
for alvo in config.TARGETS:
    preds = carregar(alvo)
    for m, df in preds.items():
        t = df[df["conjunto"] == "teste"]
        if t.empty:
            continue
        H = sorted(int(c.split("_")[1][:-1]) for c in t.columns if c.startswith("pred_"))
        obs = np.column_stack([t[f"obs_{h}h"] for h in H]); prd = np.column_stack([t[f"pred_{h}h"] for h in H])
        tab = metrics.summary_table(obs, prd, t["cota_t"].to_numpy(), H).reset_index()
        tab.insert(0, "modelo", m); tab.insert(0, "estacao", alvo); linhas.append(tab)
geral = pd.concat(linhas).set_index(["estacao", "modelo", "horizonte_h"])
display(geral)
geral.to_csv(config.OUTPUTS / f"resultados_teste_{pipeline.current_tag()}.csv")

## 2. Por evento: pico, horário do pico e alerta

In [ ]:
linhas = []
for alvo in config.TARGETS:
    preds = carregar(alvo)
    limiar = config.ALERT_LEVEL_CM.get(alvo)
    for m, df in preds.items():
        for h in config.HORIZONS:
            obs = metrics.as_series(df.index, df[f"obs_{h}h"], h)
            prd = metrics.as_series(df.index, df[f"pred_{h}h"], h)
            for ev, (ini, fim) in config.EVENTS.items():
                st = metrics.event_stats(obs, prd, ini, fim)
                if not st:
                    continue
                if limiar is not None:
                    st.update(metrics.alert_scores(obs.loc[ini:fim], prd.loc[ini:fim], limiar))
                linhas.append({"estacao": alvo, "modelo": m, "horizonte_h": h, "evento": ev, **st})
eventos = pd.DataFrame(linhas).set_index(["estacao", "evento", "horizonte_h", "modelo"]).round(2)
display(eventos)
eventos.to_csv(config.OUTPUTS / f"resultados_eventos_{pipeline.current_tag()}.csv")
if all(v is None for v in config.ALERT_LEVEL_CM.values()):
    print("Obs.: ALERT_LEVEL_CM vazio em config.py -> colunas de alerta (POD/FAR/CSI) não calculadas.")

## 3. Gráficos: observado × previsto

In [ ]:
cores = {"persistencia": "0.55", "ridge": "tab:green", "hgb": "tab:orange", "mlp": "tab:purple", "gru": "tab:red"}
for alvo in config.TARGETS:
    preds = carregar(alvo)
    if not preds:
        continue
    for ev, (ini, fim) in config.EVENTS.items():
        fig, axes = plt.subplots(len(config.HORIZONS), 1, figsize=(11, 3.0 * len(config.HORIZONS)), sharex=True)
        for ax, h in zip(np.atleast_1d(axes), config.HORIZONS):
            ref = next(iter(preds.values()))
            obs = metrics.as_series(ref.index, ref[f"obs_{h}h"], h).loc[ini:fim]
            if obs.empty:
                continue
            ax.plot(obs.index, obs, color="k", lw=2, label="observado")
            for m, df in preds.items():
                s = metrics.as_series(df.index, df[f"pred_{h}h"], h).loc[ini:fim]
                ax.plot(s.index, s, lw=1.2, color=cores[m], ls="--" if m == "persistencia" else "-", label=m)
            ax.set_ylabel(f"cota (cm)\nprevisão {h} h")
        np.atleast_1d(axes)[0].legend(ncol=6, fontsize=8)
        np.atleast_1d(axes)[0].set_title(f"{config.STATIONS[alvo]['nome']} — {ev}")
        plt.tight_layout()
        fig.savefig(FIG_DIR / f"evento_{alvo}_{ev.replace('/', '-')}.png", dpi=150)
        plt.show()

## 4. Com × sem IMERG (rode os notebooks 03–05 nos dois cenários antes)

Compara, **nos mesmos instantes**, o erro dos modelos sem e com a chuva do IMERG. `delta_RMSE < 0` = o IMERG ajudou.
Para gerar o cenário com IMERG: `USE_IMERG = True` em `config.py`, reinicie o kernel e rode 03 → 05 de novo.

In [ ]:
linhas = []
for alvo in config.TARGETS:
    for m in MODELOS:
        try:
            a = pipeline.load_predictions(m, alvo, tag="sem_imerg"); b = pipeline.load_predictions(m, alvo, tag="com_imerg")
        except FileNotFoundError:
            continue
        idx = a.index.intersection(b.index)
        a, b = a.loc[idx], b.loc[idx]
        a, b = a[a["conjunto"] == "teste"], b[b["conjunto"] == "teste"]
        for h in config.HORIZONS:
            o = a[f"obs_{h}h"]
            linhas.append({"estacao": alvo, "modelo": m, "horizonte_h": h, "n": len(a),
                           "RMSE_sem": metrics.rmse(o, a[f"pred_{h}h"]), "RMSE_com": metrics.rmse(o, b[f"pred_{h}h"])})
if linhas:
    comp = pd.DataFrame(linhas); comp["delta_RMSE_cm"] = comp["RMSE_com"] - comp["RMSE_sem"]
    comp["delta_%"] = 100 * comp["delta_RMSE_cm"] / comp["RMSE_sem"]
    display(comp.set_index(["estacao", "modelo", "horizonte_h"]).round(2))
    comp.to_csv(config.OUTPUTS / "comparacao_com_sem_imerg.csv", index=False)
else:
    print("Ainda falta rodar os dois cenários (sem_imerg e com_imerg).")

## 5. O que levar para o artigo

- **Tabela 1** (`resultados_teste_<cenário>.csv`): RMSE/MAE/NSE/skill por modelo e horizonte.
- **Tabela 2** (`resultados_eventos_<cenário>.csv`): erro de pico e de horário por evento.
- **Figuras** em `outputs/figs/<cenário>/`.
- **Discussão:** quanto cada modelo subestima o pico em mai/2024 (acima do histórico de treino) e em que horizonte o ganho sobre a persistência desaparece.
- Se tiver boletins do SAH-Taquari dos eventos, compare **qualitativamente** as previsões oficiais de 4–6 h com as suas.